# MoNuSAC 数据集预处理

本 notebook 包含三条流水线（路径均相对 `contrast/HoverNet/`）：

1. **边界框（检测）**：输出到 `../data/MoNuSac`，512×512 patch、20× 倍率、Faster R-CNN 用 `boxes.csv`。
2. **点标注**：输出到 `../data/MoNuSAC_point`，512×512 patch、可调倍率（推荐 **5×** 或 **1×**）、`points.csv` + `labels/{split}/*.mat`（CoNSeP 风格 `inst_centroid` / `inst_type` / `inst_map` / `type_map`）；仅保留每 patch **200–400** 个细胞且 **≥2** 种类别的 patch。
3. **分割掩码（HoverNet）**：输出到 `../data/MoNuSAC_seg`，与点标注 **完全相同的 patch 与倍率**，目录结构 `{split}/Images/` + `{split}/Labels/*.mat`，可直接作为 `extract_patches.py` 输入。

原始数据目录：`../data/MoNuSAC_raw`。

边缘或整图小于 512 时，**缩放**到 512×512（不补黑边）；标注坐标会按同样比例映射到输出图像。

患者级 train/val/test 划分与随机种子各流水线共用。

In [1]:
from __future__ import annotations

import csv
import math
import random
import shutil
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
from tqdm.auto import tqdm

from monusac_slide_reader import SlideReader, choose_best_level
from monusac_label_utils import (
    build_point_label_mat,
    build_seg_label_mat,
    empty_label_mat,
    ensure_point_label_dirs,
    ensure_seg_output_root,
    parse_seg_annotations,
    patch_seg_annotations,
    save_label_mat,
    vertices_to_output_patch,
)

# 运行前请确保已安装依赖：Pillow、tqdm、opencv-python、scipy；
# 若需要直接读取 .svs，还需要 openslide-python 和 openslide 二进制库。

SOURCE_ROOT = Path("../data/MoNuSAC_raw")
OUTPUT_ROOT = Path("../data/MoNuSac")
OUTPUT_POINT_ROOT = Path("../data/MoNuSAC_point")
OUTPUT_SEG_ROOT = Path("../data/MoNuSAC_seg")
PATCH_SIZE = 512
STRIDE = 256
TARGET_MAGNIFICATION = 20.0
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15
NEGATIVE_RATIO = 0.25
MIN_TISSUE_FRACTION = 0.05
WHITE_THRESHOLD = 220
SEED = 42
OVERWRITE_OUTPUT = False

# 点标注流水线参数（5× 或 1× 均可；1× 视野更大、细胞更密，但单细胞像素更小）
POINT_TARGET_MAGNIFICATION = 5.0
TIF_REFERENCE_MAGNIFICATION = 40.0
MIN_CELLS_PER_PATCH = 200
MAX_CELLS_PER_PATCH = 400
MIN_DISTINCT_LABELS_PER_PATCH = 2
OVERWRITE_POINT_OUTPUT = False
OVERWRITE_SEG_OUTPUT = False

VALID_IMAGE_SUFFIXES = (".tif", ".tiff", ".svs")
OBJECTIVE_PROPERTY_KEYS = (
    "openslide.objective-power",
    "aperio.AppMag",
    "hamamatsu.SourceLens",
)


@dataclass(frozen=True)
class RegionAnnotation:
    label: str
    bbox: tuple[int, int, int, int]


@dataclass(frozen=True)
class PointRegionAnnotation:
    label: str
    bbox: tuple[int, int, int, int]
    centroid: tuple[float, float]


@dataclass(frozen=True)
class SlideSample:
    patient_id: str
    sample_id: str
    image_path: Path
    xml_path: Path

e:\WYH\CV\MCSpatNet\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# SlideReader / choose_best_level 见 monusac_slide_reader.py（TIFF 用 OpenCV 读取，避免 Pillow 崩溃）


def collect_samples(source_root: Path) -> list[SlideSample]:
    suffix_priority = {".tif": 0, ".tiff": 0, ".svs": 1}
    samples: list[SlideSample] = []
    for patient_dir in sorted(path for path in source_root.iterdir() if path.is_dir()):
        image_candidates: dict[str, Path] = {}
        for image_path in sorted(patient_dir.iterdir()):
            if image_path.suffix.lower() not in suffix_priority:
                continue
            stem = image_path.stem
            current = image_candidates.get(stem)
            if current is None or suffix_priority[image_path.suffix.lower()] < suffix_priority[current.suffix.lower()]:
                image_candidates[stem] = image_path

        for xml_path in sorted(patient_dir.glob("*.xml")):
            sample_id = xml_path.stem
            image_path = image_candidates.get(sample_id)
            if image_path is None:
                raise FileNotFoundError(f"标注文件缺少对应图像: {xml_path}")
            samples.append(
                SlideSample(
                    patient_id=patient_dir.name,
                    sample_id=sample_id,
                    image_path=image_path,
                    xml_path=xml_path,
                )
            )

    if not samples:
        raise FileNotFoundError(f"在 {source_root} 中未找到 MoNuSAC 样本。")

    return samples


def parse_annotations(xml_path: Path) -> list[RegionAnnotation]:
    root = ET.parse(xml_path).getroot()
    annotations: list[RegionAnnotation] = []
    for annotation in root.findall("Annotation"):
        attribute = annotation.find("./Attributes/Attribute")
        label = attribute.attrib.get("Name", "Unknown") if attribute is not None else "Unknown"
        for region in annotation.findall("./Regions/Region"):
            vertices = region.findall("./Vertices/Vertex")
            xs = [float(vertex.attrib["X"]) for vertex in vertices if "X" in vertex.attrib]
            ys = [float(vertex.attrib["Y"]) for vertex in vertices if "Y" in vertex.attrib]
            if not xs or not ys:
                continue
            xmin = max(0, int(math.floor(min(xs))))
            ymin = max(0, int(math.floor(min(ys))))
            xmax = int(math.ceil(max(xs)))
            ymax = int(math.ceil(max(ys)))
            if xmax <= xmin or ymax <= ymin:
                continue
            annotations.append(RegionAnnotation(label=label, bbox=(xmin, ymin, xmax, ymax)))
    return annotations


def parse_point_annotations(
    xml_path: Path, coordinate_scale: float = 1.0
) -> list[PointRegionAnnotation]:
    root = ET.parse(xml_path).getroot()
    annotations: list[PointRegionAnnotation] = []
    for annotation in root.findall("Annotation"):
        attribute = annotation.find("./Attributes/Attribute")
        label = attribute.attrib.get("Name", "Unknown") if attribute is not None else "Unknown"
        for region in annotation.findall("./Regions/Region"):
            vertices = region.findall("./Vertices/Vertex")
            xs = [float(vertex.attrib["X"]) for vertex in vertices if "X" in vertex.attrib]
            ys = [float(vertex.attrib["Y"]) for vertex in vertices if "Y" in vertex.attrib]
            if not xs or not ys:
                continue
            scaled_xs = [x * coordinate_scale for x in xs]
            scaled_ys = [y * coordinate_scale for y in ys]
            xmin = max(0, int(math.floor(min(scaled_xs))))
            ymin = max(0, int(math.floor(min(scaled_ys))))
            xmax = int(math.ceil(max(scaled_xs)))
            ymax = int(math.ceil(max(scaled_ys)))
            if xmax <= xmin or ymax <= ymin:
                continue
            centroid_x = sum(scaled_xs) / len(scaled_xs)
            centroid_y = sum(scaled_ys) / len(scaled_ys)
            annotations.append(
                PointRegionAnnotation(
                    label=label,
                    bbox=(xmin, ymin, xmax, ymax),
                    centroid=(centroid_x, centroid_y),
                )
            )
    return annotations


def annotation_coordinate_scale(
    image_path: Path,
    reader: SlideReader,
    target_magnification: float,
    tif_reference_magnification: float,
) -> float:
    if image_path.suffix.lower() in {".tif", ".tiff"}:
        if tif_reference_magnification <= 0 or target_magnification <= 0:
            return 1.0
        return target_magnification / tif_reference_magnification
    if reader._downsample > 0:
        return 1.0 / reader._downsample
    return 1.0

In [3]:
def split_patients(
    patient_ids: Iterable[str],
    train_ratio: float,
    val_ratio: float,
    test_ratio: float,
    seed: int,
 ) -> dict[str, str]:
    ratios = [train_ratio, val_ratio, test_ratio]
    if any(r <= 0 for r in ratios) or not math.isclose(sum(ratios), 1.0, rel_tol=1e-6, abs_tol=1e-6):
        raise ValueError("train/val/test 比例必须都大于 0，且总和为 1.0")

    patients = sorted(set(patient_ids))
    rng = random.Random(seed)
    rng.shuffle(patients)

    total = len(patients)
    train_count = max(1, round(total * train_ratio))
    val_count = max(1, round(total * val_ratio)) if total >= 3 else max(0, total - train_count - 1)

    if train_count + val_count >= total:
        val_count = max(1, total - train_count - 1) if total >= 3 else 0

    test_count = total - train_count - val_count
    if test_count <= 0 and total >= 3:
        test_count = 1
        if train_count > val_count:
            train_count -= 1
        else:
            val_count = max(1, val_count - 1)

    boundaries = {
        "train": patients[:train_count],
        "val": patients[train_count : train_count + val_count],
        "test": patients[train_count + val_count :],
    }

    mapping: dict[str, str] = {}
    for split_name, split_patients_list in boundaries.items():
        for patient_id in split_patients_list:
            mapping[patient_id] = split_name
    return mapping


def sliding_positions(length: int, patch_size: int, stride: int) -> list[int]:
    if length <= patch_size:
        return [0]
    positions = list(range(0, length - patch_size + 1, stride))
    if positions[-1] != length - patch_size:
        positions.append(length - patch_size)
    return positions


def clip_box_to_patch(
    bbox: tuple[int, int, int, int], patch_x: int, patch_y: int, patch_size: int
 ) -> tuple[int, int, int, int] | None:
    xmin, ymin, xmax, ymax = bbox
    clipped = (
        max(0, xmin - patch_x),
        max(0, ymin - patch_y),
        min(patch_size, xmax - patch_x),
        min(patch_size, ymax - patch_y),
    )
    if clipped[2] <= clipped[0] or clipped[3] <= clipped[1]:
        return None
    return clipped


def bbox_center_in_patch(
    bbox: tuple[int, int, int, int], patch_x: int, patch_y: int, patch_size: int
 ) -> bool:
    xmin, ymin, xmax, ymax = bbox
    center_x = (xmin + xmax) / 2.0
    center_y = (ymin + ymax) / 2.0
    return patch_x <= center_x < patch_x + patch_size and patch_y <= center_y < patch_y + patch_size


def centroid_in_patch(
    centroid: tuple[float, float], patch_x: int, patch_y: int, patch_size: int
) -> bool:
    center_x, center_y = centroid
    return (
        patch_x <= center_x < patch_x + patch_size
        and patch_y <= center_y < patch_y + patch_size
    )


def patch_point_annotations(
    annotations: list[PointRegionAnnotation], patch_x: int, patch_y: int, patch_size: int
) -> list[tuple[PointRegionAnnotation, float, float]]:
    patch_points: list[tuple[PointRegionAnnotation, float, float]] = []
    for annotation in annotations:
        if not centroid_in_patch(annotation.centroid, patch_x, patch_y, patch_size):
            continue
        rel_x = annotation.centroid[0] - patch_x
        rel_y = annotation.centroid[1] - patch_y
        patch_points.append((annotation, rel_x, rel_y))
    return patch_points


def patch_meets_point_criteria(
    patch_points: list[tuple[PointRegionAnnotation, float, float]],
    min_cells: int,
    max_cells: int,
    min_distinct_labels: int,
) -> bool:
    if not (min_cells <= len(patch_points) <= max_cells):
        return False
    distinct_labels = {annotation.label for annotation, _, _ in patch_points}
    return len(distinct_labels) >= min_distinct_labels


def estimate_tissue_fraction(image, white_threshold: int) -> float:
    grayscale = image.convert("L")
    pixels = grayscale.load()
    width, height = grayscale.size
    tissue_pixels = 0
    total_pixels = width * height
    for y in range(height):
        for x in range(width):
            if pixels[x, y] < white_threshold:
                tissue_pixels += 1
    return tissue_pixels / max(total_pixels, 1)


def scale_box_to_output_patch(
    bbox: tuple[int, int, int, int], scale_x: float, scale_y: float
) -> tuple[float, float, float, float]:
    xmin, ymin, xmax, ymax = bbox
    return (xmin * scale_x, ymin * scale_y, xmax * scale_x, ymax * scale_y)


def scale_point_to_output_patch(x: float, y: float, scale_x: float, scale_y: float) -> tuple[float, float]:
    return x * scale_x, y * scale_y


def ensure_output_root(output_root: Path, overwrite: bool) -> None:
    if output_root.exists() and overwrite:
        shutil.rmtree(output_root)

    (output_root / "images").mkdir(parents=True, exist_ok=True)
    (output_root / "annotations").mkdir(parents=True, exist_ok=True)
    (output_root / "metadata").mkdir(parents=True, exist_ok=True)

    for split_name in ("train", "val", "test"):
        (output_root / "images" / split_name).mkdir(parents=True, exist_ok=True)

In [4]:
def prepare_monusac_dataset(
    source_root: Path = SOURCE_ROOT,
    output_root: Path = OUTPUT_ROOT,
    patch_size: int = PATCH_SIZE,
    stride: int = STRIDE,
    target_magnification: float = TARGET_MAGNIFICATION,
    train_ratio: float = TRAIN_RATIO,
    val_ratio: float = VAL_RATIO,
    test_ratio: float = TEST_RATIO,
    negative_ratio: float = NEGATIVE_RATIO,
    min_tissue_fraction: float = MIN_TISSUE_FRACTION,
    white_threshold: int = WHITE_THRESHOLD,
    seed: int = SEED,
    overwrite: bool = OVERWRITE_OUTPUT,
 ) -> None:
    source_root = Path(source_root)
    output_root = Path(output_root)

    ensure_output_root(output_root, overwrite)
    samples = collect_samples(source_root)
    patient_split = split_patients(
        (sample.patient_id for sample in samples),
        train_ratio=train_ratio,
        val_ratio=val_ratio,
        test_ratio=test_ratio,
        seed=seed,
    )

    boxes_csv_path = output_root / "annotations" / "boxes.csv"
    class_csv_path = output_root / "metadata" / "classes.csv"

    fieldnames = [
        "split",
        "patient_id",
        "sample_id",
        "image_path",
        "patch_x",
        "patch_y",
        "patch_width",
        "patch_height",
        "xmin",
        "ymin",
        "xmax",
        "ymax",
        "label",
        "label_id",
        "is_negative",
    ]

    class_to_id: dict[str, int] = {}
    all_rows: list[dict[str, object]] = []
    rng = random.Random(seed)

    sample_progress = tqdm(samples, desc="处理样本", unit="sample")
    for sample in sample_progress:
        split_name = patient_split[sample.patient_id]
        sample_progress.set_postfix_str(f"{sample.sample_id} -> {split_name}")
        annotations = parse_annotations(sample.xml_path)

        for label in sorted({annotation.label for annotation in annotations}):
            class_to_id.setdefault(label, len(class_to_id) + 1)

        reader = SlideReader(sample.image_path, target_magnification)
        positive_candidates: list[tuple[int, int, list[tuple[RegionAnnotation, tuple[int, int, int, int]]]]] = []
        negative_candidates: list[tuple[int, int]] = []

        try:
            x_positions = sliding_positions(reader.width, patch_size, stride)
            y_positions = sliding_positions(reader.height, patch_size, stride)

            for patch_y in y_positions:
                for patch_x in x_positions:
                    patch_annotations: list[tuple[RegionAnnotation, tuple[int, int, int, int]]] = []
                    for annotation in annotations:
                        if not bbox_center_in_patch(annotation.bbox, patch_x, patch_y, patch_size):
                            continue
                        clipped_box = clip_box_to_patch(annotation.bbox, patch_x, patch_y, patch_size)
                        if clipped_box is None:
                            continue
                        patch_annotations.append((annotation, clipped_box))

                    if patch_annotations:
                        positive_candidates.append((patch_x, patch_y, patch_annotations))
                    else:
                        negative_candidates.append((patch_x, patch_y))

            negative_limit = int(len(positive_candidates) * negative_ratio)
            rng.shuffle(negative_candidates)
            kept_negatives = 0

            for patch_x, patch_y, patch_annotations in positive_candidates:
                patch_image, scale_x, scale_y = reader.crop(patch_x, patch_y, patch_size)
                relative_path = Path("images") / split_name / f"{sample.sample_id}__x{patch_x}_y{patch_y}.png"
                patch_image.save(output_root / relative_path)

                for annotation, clipped_box in patch_annotations:
                    xmin, ymin, xmax, ymax = scale_box_to_output_patch(
                        clipped_box, scale_x, scale_y
                    )
                    row = {
                        "split": split_name,
                        "patient_id": sample.patient_id,
                        "sample_id": sample.sample_id,
                        "image_path": relative_path.as_posix(),
                        "patch_x": patch_x,
                        "patch_y": patch_y,
                        "patch_width": patch_size,
                        "patch_height": patch_size,
                        "xmin": round(xmin, 2),
                        "ymin": round(ymin, 2),
                        "xmax": round(xmax, 2),
                        "ymax": round(ymax, 2),
                        "label": annotation.label,
                        "label_id": class_to_id[annotation.label],
                        "is_negative": 0,
                    }
                    all_rows.append(row)

            for patch_x, patch_y in negative_candidates:
                if kept_negatives >= negative_limit:
                    break
                patch_image, _, _ = reader.crop(patch_x, patch_y, patch_size)
                tissue_fraction = estimate_tissue_fraction(patch_image, white_threshold)
                if tissue_fraction < min_tissue_fraction:
                    continue

                relative_path = Path("images") / split_name / f"{sample.sample_id}__x{patch_x}_y{patch_y}.png"
                patch_image.save(output_root / relative_path)

                row = {
                    "split": split_name,
                    "patient_id": sample.patient_id,
                    "sample_id": sample.sample_id,
                    "image_path": relative_path.as_posix(),
                    "patch_x": patch_x,
                    "patch_y": patch_y,
                    "patch_width": patch_size,
                    "patch_height": patch_size,
                    "xmin": "",
                    "ymin": "",
                    "xmax": "",
                    "ymax": "",
                    "label": "background",
                    "label_id": 0,
                    "is_negative": 1,
                }
                all_rows.append(row)
                kept_negatives += 1

        finally:
            reader.close()

    with boxes_csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)

    with class_csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["label_id", "label"])
        writer.writerow([0, "background"])
        for label, label_id in sorted(class_to_id.items(), key=lambda item: item[1]):
            writer.writerow([label_id, label])

    print(f"处理完成，输出目录: {output_root.resolve()}")
    print(f"总样本数: {len(samples)}")
    print(f"类别数(不含背景): {len(class_to_id)}")

In [5]:
def prepare_monusac_point_and_seg_dataset(
    source_root: Path = SOURCE_ROOT,
    output_root: Path = OUTPUT_POINT_ROOT,
    seg_output_root: Path = OUTPUT_SEG_ROOT,
    patch_size: int = PATCH_SIZE,
    stride: int = STRIDE,
    target_magnification: float = POINT_TARGET_MAGNIFICATION,
    tif_reference_magnification: float = TIF_REFERENCE_MAGNIFICATION,
    min_cells_per_patch: int = MIN_CELLS_PER_PATCH,
    max_cells_per_patch: int = MAX_CELLS_PER_PATCH,
    min_distinct_labels_per_patch: int = MIN_DISTINCT_LABELS_PER_PATCH,
    train_ratio: float = TRAIN_RATIO,
    val_ratio: float = VAL_RATIO,
    test_ratio: float = TEST_RATIO,
    negative_ratio: float = NEGATIVE_RATIO,
    min_tissue_fraction: float = MIN_TISSUE_FRACTION,
    white_threshold: int = WHITE_THRESHOLD,
    seed: int = SEED,
    overwrite: bool = OVERWRITE_POINT_OUTPUT,
    overwrite_seg: bool = OVERWRITE_SEG_OUTPUT,
) -> None:
    source_root = Path(source_root)
    output_root = Path(output_root)
    seg_output_root = Path(seg_output_root)

    ensure_output_root(output_root, overwrite)
    ensure_point_label_dirs(output_root)
    ensure_seg_output_root(seg_output_root, overwrite_seg)
    samples = collect_samples(source_root)
    patient_split = split_patients(
        (sample.patient_id for sample in samples),
        train_ratio=train_ratio,
        val_ratio=val_ratio,
        test_ratio=test_ratio,
        seed=seed,
    )

    points_csv_path = output_root / "annotations" / "points.csv"
    class_csv_path = output_root / "metadata" / "classes.csv"

    fieldnames = [
        "split",
        "patient_id",
        "sample_id",
        "image_path",
        "patch_x",
        "patch_y",
        "patch_width",
        "patch_height",
        "x",
        "y",
        "label",
        "label_id",
        "is_negative",
    ]

    class_to_id: dict[str, int] = {}
    all_rows: list[dict[str, object]] = []
    rng = random.Random(seed)
    kept_patches = 0
    skipped_patches = 0

    sample_progress = tqdm(samples, desc="点标注处理", unit="sample")
    for sample in sample_progress:
        split_name = patient_split[sample.patient_id]
        sample_progress.set_postfix_str(f"{sample.sample_id} -> {split_name}")

        reader = SlideReader(
            sample.image_path,
            target_magnification,
            tif_reference_magnification=tif_reference_magnification,
        )
        coord_scale = annotation_coordinate_scale(
            sample.image_path,
            reader,
            target_magnification,
            tif_reference_magnification,
        )
        annotations = parse_point_annotations(sample.xml_path, coordinate_scale=coord_scale)
        seg_annotations = parse_seg_annotations(sample.xml_path, coordinate_scale=coord_scale)

        for label in sorted({annotation.label for annotation in annotations}):
            class_to_id.setdefault(label, len(class_to_id) + 1)

        positive_candidates: list[
            tuple[int, int, list[tuple[PointRegionAnnotation, float, float]]]
        ] = []
        negative_candidates: list[tuple[int, int]] = []

        try:
            x_positions = sliding_positions(reader.width, patch_size, stride)
            y_positions = sliding_positions(reader.height, patch_size, stride)

            for patch_y in y_positions:
                for patch_x in x_positions:
                    patch_points = patch_point_annotations(
                        annotations, patch_x, patch_y, patch_size
                    )
                    if patch_meets_point_criteria(
                        patch_points,
                        min_cells=min_cells_per_patch,
                        max_cells=max_cells_per_patch,
                        min_distinct_labels=min_distinct_labels_per_patch,
                    ):
                        positive_candidates.append((patch_x, patch_y, patch_points))
                    else:
                        if not patch_points:
                            negative_candidates.append((patch_x, patch_y))
                        skipped_patches += 1

            negative_limit = int(len(positive_candidates) * negative_ratio)
            rng.shuffle(negative_candidates)
            kept_negatives = 0

            for patch_x, patch_y, patch_points in positive_candidates:
                patch_image, scale_x, scale_y = reader.crop(patch_x, patch_y, patch_size)
                patch_stem = f"{sample.sample_id}__x{patch_x}_y{patch_y}"
                relative_path = Path("images") / split_name / f"{patch_stem}.png"
                patch_image.save(output_root / relative_path)
                patch_image.save(seg_output_root / split_name / "Images" / f"{patch_stem}.png")
                kept_patches += 1

                centroids_xy: list[tuple[float, float]] = []
                label_ids: list[int] = []
                for annotation, rel_x, rel_y in patch_points:
                    out_x, out_y = scale_point_to_output_patch(rel_x, rel_y, scale_x, scale_y)
                    centroids_xy.append((out_x, out_y))
                    label_ids.append(class_to_id[annotation.label])
                    all_rows.append(
                        {
                            "split": split_name,
                            "patient_id": sample.patient_id,
                            "sample_id": sample.sample_id,
                            "image_path": relative_path.as_posix(),
                            "patch_x": patch_x,
                            "patch_y": patch_y,
                            "patch_width": patch_size,
                            "patch_height": patch_size,
                            "x": round(out_x, 2),
                            "y": round(out_y, 2),
                            "label": annotation.label,
                            "label_id": class_to_id[annotation.label],
                            "is_negative": 0,
                        }
                    )

                point_mat = build_point_label_mat(centroids_xy, label_ids, patch_size, patch_size)
                save_label_mat(output_root / "labels" / split_name / f"{patch_stem}.mat", point_mat)

                patch_regions = patch_seg_annotations(seg_annotations, patch_x, patch_y, patch_size)
                seg_regions: list[tuple[int, np.ndarray]] = []
                for region in patch_regions:
                    polygon = vertices_to_output_patch(
                        region.vertices,
                        patch_x,
                        patch_y,
                        scale_x,
                        scale_y,
                        patch_size,
                    )
                    seg_regions.append((class_to_id[region.label], polygon))
                seg_mat = build_seg_label_mat(seg_regions, patch_size, patch_size)
                save_label_mat(
                    seg_output_root / split_name / "Labels" / f"{patch_stem}.mat",
                    seg_mat,
                )

            for patch_x, patch_y in negative_candidates:
                if kept_negatives >= negative_limit:
                    break
                patch_image, _, _ = reader.crop(patch_x, patch_y, patch_size)
                tissue_fraction = estimate_tissue_fraction(patch_image, white_threshold)
                if tissue_fraction < min_tissue_fraction:
                    continue

                patch_stem = f"{sample.sample_id}__x{patch_x}_y{patch_y}"
                relative_path = Path("images") / split_name / f"{patch_stem}.png"
                patch_image.save(output_root / relative_path)
                patch_image.save(seg_output_root / split_name / "Images" / f"{patch_stem}.png")
                kept_patches += 1

                empty_mat = empty_label_mat(patch_size, patch_size)
                save_label_mat(output_root / "labels" / split_name / f"{patch_stem}.mat", empty_mat)
                save_label_mat(seg_output_root / split_name / "Labels" / f"{patch_stem}.mat", empty_mat)

                all_rows.append(
                    {
                        "split": split_name,
                        "patient_id": sample.patient_id,
                        "sample_id": sample.sample_id,
                        "image_path": relative_path.as_posix(),
                        "patch_x": patch_x,
                        "patch_y": patch_y,
                        "patch_width": patch_size,
                        "patch_height": patch_size,
                        "x": "",
                        "y": "",
                        "label": "background",
                        "label_id": 0,
                        "is_negative": 1,
                    }
                )
                kept_negatives += 1

        finally:
            reader.close()

    with points_csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)

    with class_csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["label_id", "label"])
        writer.writerow([0, "background"])
        for label, label_id in sorted(class_to_id.items(), key=lambda item: item[1]):
            writer.writerow([label_id, label])

    positive_images = len({row["image_path"] for row in all_rows if int(row["is_negative"]) == 0})
    print(f"点标注处理完成，输出目录: {output_root.resolve()}")
    print(f"分割掩码输出目录: {seg_output_root.resolve()}")
    print(f"目标倍率: {target_magnification}x (TIF 参考倍率: {tif_reference_magnification}x)")
    print(f"保留 patch 数: {kept_patches} (正样本图: {positive_images})")
    print(f"跳过/未入选 patch 窗口数: {skipped_patches}")
    print(f"点标注条数: {sum(1 for row in all_rows if int(row['is_negative']) == 0)}")
    print(f"类别数(不含背景): {len(class_to_id)}")
    print(
        "extract_patches.py 可配置: "
        f"img=../data/MoNuSAC_seg/{{split}}/Images/, ann=../data/MoNuSAC_seg/{{split}}/Labels/"
    )


# 兼容旧函数名
prepare_monusac_point_dataset = prepare_monusac_point_and_seg_dataset

In [6]:
# 边界框流水线：按需修改参数后运行。

prepare_monusac_dataset(
    source_root=SOURCE_ROOT,
    output_root=OUTPUT_ROOT,
    patch_size=PATCH_SIZE,
    stride=STRIDE,
    target_magnification=TARGET_MAGNIFICATION,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    negative_ratio=NEGATIVE_RATIO,
    min_tissue_fraction=MIN_TISSUE_FRACTION,
    white_threshold=WHITE_THRESHOLD,
    seed=SEED,
    overwrite=OVERWRITE_OUTPUT,
)

处理样本: 100%|██████████| 209/209 [02:55<00:00,  1.19sample/s, TCGA-YL-A9WY-01Z-00-DX1-5 -> train] 


处理完成，输出目录: E:\WYH\CV\MCSpatNet\contrast\data\MoNuSac
总样本数: 209
类别数(不含背景): 4


In [7]:
# 点标注 + 分割掩码流水线：将 POINT_TARGET_MAGNIFICATION 设为 5.0 或 1.0 后运行。

prepare_monusac_point_and_seg_dataset(
    source_root=SOURCE_ROOT,
    output_root=OUTPUT_POINT_ROOT,
    seg_output_root=OUTPUT_SEG_ROOT,
    patch_size=PATCH_SIZE,
    stride=STRIDE,
    target_magnification=POINT_TARGET_MAGNIFICATION,
    tif_reference_magnification=TIF_REFERENCE_MAGNIFICATION,
    min_cells_per_patch=MIN_CELLS_PER_PATCH,
    max_cells_per_patch=MAX_CELLS_PER_PATCH,
    min_distinct_labels_per_patch=MIN_DISTINCT_LABELS_PER_PATCH,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    negative_ratio=NEGATIVE_RATIO,
    min_tissue_fraction=MIN_TISSUE_FRACTION,
    white_threshold=WHITE_THRESHOLD,
    seed=SEED,
    overwrite=OVERWRITE_POINT_OUTPUT,
    overwrite_seg=OVERWRITE_SEG_OUTPUT,
)

点标注处理: 100%|██████████| 209/209 [01:13<00:00,  2.85sample/s, TCGA-YL-A9WY-01Z-00-DX1-5 -> train] 


点标注处理完成，输出目录: E:\WYH\CV\MCSpatNet\contrast\data\MoNuSAC_point
分割掩码输出目录: E:\WYH\CV\MCSpatNet\contrast\data\MoNuSAC_seg
目标倍率: 5.0x (TIF 参考倍率: 40.0x)
保留 patch 数: 45 (正样本图: 45)
跳过/未入选 patch 窗口数: 164
点标注条数: 13132
类别数(不含背景): 4
extract_patches.py 可配置: img=../data/MoNuSAC_seg/{split}/Images/, ann=../data/MoNuSAC_seg/{split}/Labels/
